<a href="https://colab.research.google.com/github/SilviaPosso/Yahtzee-Simulaci-n/blob/main/Posso_Mazo_Silvia_Rosa_Montecarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
╔══════════════════════════════════════════════════════╗
║        YAHTZEE CLÁSICO — SIMULACIÓN MONTECARLO       ║
║                 2 Jugadores | 5 Dados |
╚══════════════════════════════════════════════════════╝

"""

import random
import collections

# ══════════════════════════════════════════════════════
# CONSTANTES
# ══════════════════════════════════════════════════════

NUM_DADOS  = 5
MAX_LANZAM = 3
CARAS      = 6

# Todas las categorías del juego, en orden de presentación
CATEGORIAS = [
    "Unos", "Doses", "Treses", "Cuatros", "Cincos", "Seises",
    "Escalera corta", "Escalera larga", "Full House",
    "Póker (4 iguales)", "Yahtzee (5 iguales)", "Azar",
]

# Emojis para mostrar cada cara del dado
CARA_EMOJI = {1: "⚀", 2: "⚁", 3: "⚂", 4: "⚃", 5: "⚄", 6: "⚅"}


# ══════════════════════════════════════════════════════
# FUNCIONES DE DADOS
# ══════════════════════════════════════════════════════

def lanzar_dados(n=NUM_DADOS):
    """
    Simula el lanzamiento de n dados de 6 caras.
    Distribución uniforme: cada cara tiene probabilidad 1/6.
    """
    return [random.randint(1, CARAS) for _ in range(n)]


def mostrar_dados(dados):
    """Devuelve los dados como cadena de emojis y números."""
    emojis  = " ".join(CARA_EMOJI[d] for d in dados)
    numeros = " ".join(str(d) for d in dados)
    return f"{emojis}  ({numeros})"


# ══════════════════════════════════════════════════════
# CÁLCULO DE PUNTOS
# ══════════════════════════════════════════════════════

def calcular_puntos(dados, categoria):
    """
    Calcula los puntos que otorga una combinación de dados
    para la categoría indicada.

    Reglas de puntuación:
      - Unos..Seises   : suma de los dados con ese valor
      - Escalera corta : 4 valores consecutivos → 30 pts
      - Escalera larga : 5 valores consecutivos → 40 pts
      - Full House     : trío + par               → 25 pts
      - Póker          : cuatro iguales           → suma total
      - Yahtzee        : cinco iguales            → 50 pts
      - Azar           : cualquier combinación    → suma total
    """
    conteo = collections.Counter(dados)

    # ── Sección superior (Unos a Seises) ──
    numero = {"Unos": 1, "Doses": 2, "Treses": 3,
               "Cuatros": 4, "Cincos": 5, "Seises": 6}
    if categoria in numero:
        valor = numero[categoria]
        return dados.count(valor) * valor

    # ── Sección inferior ──
    if categoria == "Escalera corta":
        # Basta con que existan 4 valores consecutivos
        for inicio in range(1, 4):
            if all(v in conteo for v in range(inicio, inicio + 4)):
                return 30
        return 0

    if categoria == "Escalera larga":
        # Los 5 dados deben ser distintos y consecutivos
        valores_unicos = sorted(set(dados))
        if len(valores_unicos) == 5 and valores_unicos[-1] - valores_unicos[0] == 4:
            return 40
        return 0

    if categoria == "Full House":
        frecuencias = sorted(conteo.values())
        return 25 if frecuencias == [2, 3] else 0

    if categoria == "Póker (4 iguales)":
        return sum(dados) if max(conteo.values()) >= 4 else 0

    if categoria == "Yahtzee (5 iguales)":
        return 50 if max(conteo.values()) == 5 else 0

    if categoria == "Azar":
        return sum(dados)

    return 0  # categoría desconocida


# ══════════════════════════════════════════════════════
# LÓGICA DE LA IA
# ══════════════════════════════════════════════════════

def ia_dados_a_conservar(dados):
    """
    Estrategia de la IA: conserva los dados del valor
    que aparece con mayor frecuencia (estrategia voraz).
    En caso de empate, elige el valor más alto.
    """
    conteo  = collections.Counter(dados)
    max_frec = max(conteo.values())
    # Si hay empate en frecuencia, prefiere el dado de mayor valor
    valor_elegido = max(v for v, f in conteo.items() if f == max_frec)
    return [i for i, d in enumerate(dados) if d == valor_elegido]


def ia_mejor_categoria(dados, scorecard):
    """
    Elige la categoría disponible que otorga más puntos.
    Si ninguna da puntos, elige la primera disponible (pérdida mínima).
    """
    mejor_cat, mejor_pts = None, -1
    for cat in CATEGORIAS:
        if scorecard[cat] is None:
            pts = calcular_puntos(dados, cat)
            if pts > mejor_pts:
                mejor_pts = pts
                mejor_cat = cat
    return mejor_cat, mejor_pts


# ══════════════════════════════════════════════════════
# TURNOS
# ══════════════════════════════════════════════════════

def turno_humano(nombre, scorecard, num_turno):
    """Gestiona el turno de un jugador humano."""
    print(f"\n{'═'*54}")
    print(f"  TURNO {num_turno} — {nombre}")
    print(f"{'═'*54}")

    # Primer lanzamiento obligatorio
    dados = lanzar_dados()
    print(f"\n  Lanzamiento 1: {mostrar_dados(dados)}")

    # Hasta 2 relanzamientos opcionales
    for num_lanz in range(2, MAX_LANZAM + 1):

        # Mostrar puntuación potencial antes de decidir
        print("\n  ┌─ Puntos potenciales con estos dados ─┐")
        for cat in CATEGORIAS:
            if scorecard[cat] is None:
                pts = calcular_puntos(dados, cat)
                marcador = "★" if pts > 0 else " "
                print(f"  │ {marcador} {cat:<25} {pts:>3} pts")
        print("  └────────────────────────────────────┘")

        respuesta = input(f"\n  ¿Quieres volver a lanzar? (s/n): ").strip().lower()
        if respuesta != "s":
            break

        entrada = input("  Dados a GUARDAR — escribe los índices 1-5 "
                        "(ej: 1 3 5) o Enter para ninguno: ").strip()

        # Parsear índices del jugador (1-based → 0-based)
        indices_guardar = []
        if entrada:
            for token in entrada.split():
                if token.isdigit():
                    idx = int(token) - 1
                    if 0 <= idx < NUM_DADOS:
                        indices_guardar.append(idx)

        # Relanzar los dados NO guardados
        nuevos = lanzar_dados(NUM_DADOS - len(indices_guardar))
        dados_nuevos = []
        iter_nuevos = iter(nuevos)
        for i in range(NUM_DADOS):
            dados_nuevos.append(dados[i] if i in indices_guardar else next(iter_nuevos))
        dados = dados_nuevos

        print(f"\n  Lanzamiento {num_lanz}: {mostrar_dados(dados)}")

    # ── Elegir categoría ──
    print("\n  Elige la categoría donde anotar:")
    cats_disponibles = [cat for cat in CATEGORIAS if scorecard[cat] is None]
    for i, cat in enumerate(cats_disponibles, 1):
        pts = calcular_puntos(dados, cat)
        print(f"    {i:>2}. {cat:<25} → {pts} pts")

    while True:
        try:
            eleccion = int(input("\n  Tu elección (número): "))
            if 1 <= eleccion <= len(cats_disponibles):
                categoria_elegida = cats_disponibles[eleccion - 1]
                break
            print("  ⚠ Número fuera de rango. Intenta de nuevo.")
        except ValueError:
            print("  ⚠ Escribe un número válido.")

    puntos = calcular_puntos(dados, categoria_elegida)
    scorecard[categoria_elegida] = puntos
    print(f"\n  ✔ Anotado: {categoria_elegida} = {puntos} pts")
    return puntos


def turno_ia(nombre, scorecard, num_turno):
    """Gestiona el turno de la IA de forma automática."""
    print(f"\n{'═'*54}")
    print(f"  TURNO {num_turno} — {nombre}  🤖 (IA)")
    print(f"{'═'*54}")

    dados = lanzar_dados()
    print(f"\n  Lanzamiento 1: {mostrar_dados(dados)}")

    for num_lanz in range(2, MAX_LANZAM + 1):
        _, mejor_pts = ia_mejor_categoria(dados, scorecard)

        # Si ya tiene una combinación de 25 puntos o más, no arriesga más
        if mejor_pts >= 25:
            print(f"  → La IA decide conservar estos dados (buena mano).")
            break

        indices_guardar = ia_dados_a_conservar(dados)
        guardados = [dados[i] for i in indices_guardar]
        print(f"  → La IA guarda: {[CARA_EMOJI[d] for d in guardados]}")

        nuevos = lanzar_dados(NUM_DADOS - len(indices_guardar))
        dados_nuevos = []
        iter_nuevos  = iter(nuevos)
        for i in range(NUM_DADOS):
            dados_nuevos.append(dados[i] if i in indices_guardar else next(iter_nuevos))
        dados = dados_nuevos

        print(f"  Lanzamiento {num_lanz}: {mostrar_dados(dados)}")

    categoria, puntos = ia_mejor_categoria(dados, scorecard)
    scorecard[categoria] = puntos
    print(f"\n  ✔ La IA anota: {categoria} = {puntos} pts")
    return puntos


# ══════════════════════════════════════════════════════
# MARCADOR
# ══════════════════════════════════════════════════════

def mostrar_marcador(nombres, scorecards):
    """Imprime el marcador completo de ambos jugadores."""
    ancho_cat  = 26
    ancho_col  = 12

    print(f"\n{'─'*54}")
    print(f"  {'CATEGORÍA':<{ancho_cat}}" +
          "".join(f"{n:<{ancho_col}}" for n in nombres))
    print(f"{'─'*54}")

    for cat in CATEGORIAS:
        fila = f"  {cat:<{ancho_cat}}"
        for sc in scorecards:
            val = sc[cat]
            fila += f"{str(val) if val is not None else '—':<{ancho_col}}"
        print(fila)

    print(f"{'─'*54}")
    totales = [sum(v for v in sc.values() if v is not None) for sc in scorecards]
    print(f"  {'TOTAL':<{ancho_cat}}" +
          "".join(f"{t:<{ancho_col}}" for t in totales))
    print(f"{'─'*54}\n")


# ══════════════════════════════════════════════════════
# ESTADÍSTICAS MONTECARLO
# ══════════════════════════════════════════════════════

def estadisticas_montecarlo(n_simulaciones=10_000):
    """
    Método de Montecarlo aplicado a Yahtzee.

    Se simulan n_simulaciones lanzamientos de 5 dados y se
    cuenta cuántas veces cada categoría otorga puntos > 0.
    La frecuencia relativa converge a la probabilidad real
    conforme n_simulaciones → ∞ (Ley de los Grandes Números).

    Distribución utilizada: UNIFORME discreta en {1,2,3,4,5,6}.
    """
    frecuencias = {cat: 0 for cat in CATEGORIAS}

    for _ in range(n_simulaciones):
        dados = lanzar_dados()
        for cat in CATEGORIAS:
            if calcular_puntos(dados, cat) > 0:
                frecuencias[cat] += 1

    print(f"\n{'═'*54}")
    print(f"  ESTADÍSTICAS MONTECARLO")
    print(f"  {n_simulaciones:,} simulaciones de 5 dados")
    print(f"  Probabilidad de obtener puntos en 1 solo lanzamiento")
    print(f"{'─'*54}")

    for cat in CATEGORIAS:
        prob  = frecuencias[cat] / n_simulaciones * 100
        barra = "█" * int(prob / 2)
        print(f"  {cat:<25} {prob:5.1f}%  {barra}")

    print(f"{'═'*54}\n")
    return {cat: round(frecuencias[cat] / n_simulaciones * 100, 2) for cat in CATEGORIAS}


# ══════════════════════════════════════════════════════
# FLUJO PRINCIPAL
# ══════════════════════════════════════════════════════

def iniciar_scorecard():
    """Crea un marcador vacío (todas las categorías en None)."""
    return {cat: None for cat in CATEGORIAS}


def jugar():
    print("\n" + "═"*54)
    print("      YAHTZEE CLÁSICO — SIMULACIÓN MONTECARLO")
    print("      Distribución uniforme | Método Montecarlo")
    print("═"*54)

    # ── Seleccionar modo ──
    print("\n  Modos disponibles:")
    print("    1. Humano vs IA")
    print("    2. IA vs IA  (simulación completa)")
    modo = input("\n  Elige modo (1 o 2): ").strip()

    if modo == "2":
        nombres   = ["IA-Roja", "IA-Azul"]
        es_humano = [False, False]
    else:
        nombre = input("  Tu nombre: ").strip() or "Jugador 1"
        nombres   = [nombre, "IA"]
        es_humano = [True, False]

    scorecards = [iniciar_scorecard(), iniciar_scorecard()]
    total_turnos = len(CATEGORIAS)

    # ── Bucle principal: un turno por jugador hasta agotar categorías ──
    for num_turno in range(1, total_turnos + 1):
        for j in range(2):
            if es_humano[j]:
                turno_humano(nombres[j], scorecards[j], num_turno)
            else:
                turno_ia(nombres[j], scorecards[j], num_turno)

        mostrar_marcador(nombres, scorecards)

    # ── Resultado final ──
    totales = [sum(sc.values()) for sc in scorecards]
    print("═"*54)
    print("  RESULTADO FINAL")
    print("─"*54)
    for j in range(2):
        print(f"  {nombres[j]:<20} {totales[j]} pts")
    print("─"*54)

    if totales[0] > totales[1]:
        print(f"\n  🏆 ¡Gana {nombres[0]}!")
    elif totales[1] > totales[0]:
        print(f"\n  🏆 ¡Gana {nombres[1]}!")
    else:
        print("\n  🤝 ¡Empate!")

    print("═"*54 + "\n")

    # ── Estadísticas Montecarlo opcionales ──
    ver = input("  ¿Ver estadísticas Montecarlo? (s/n): ").strip().lower()
    if ver == "s":
        try:
            n = int(input("  Número de simulaciones (ej: 10000): ").strip())
        except ValueError:
            n = 10_000
        estadisticas_montecarlo(n)


# ══════════════════════════════════════════════════════
if __name__ == "__main__":
    jugar()